In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
sentences = [
    ("hello", "namaste"),
    ("good morning", "suprabhat"),
    ("how are you", "aap kaise ho"),
    ("i am fine", "mai theek hu"),
    ("what is your name", "tumhara naam kya hai"),
    ("my name is abhay", "mera naam abhay hai"),
    ("i love programming", "mujhe programming pasand hai"),
    ("transformers are powerful", "transformer bahut powerful hai"),
    ("deep learning is fun", "deep learning mazedar hai"),
    ("machine learning uses data", "machine learning data use karta hai"),
    ("the sun is bright", "suraj chamak raha hai"),
    ("birds can fly", "pakshi ud sakte hai"),
    ("fish live in water", "machli paani me rehti hai"),
    ("students study books", "students kitaabe padhte hai"),
    ("dogs bark loudly", "kute zor se bhokte hai"),
]

In [3]:
src_vocab = []
trg_vocab = []
input_data = []
target = []

for src, targ in sentences:
    src_vocab.extend(src.split())
    trg_vocab.extend(targ.split())

src_vocab = set(sorted(src_vocab))
trg_vocab = set(sorted(trg_vocab))

srctoi = {ch:(i) for i,ch in enumerate(src_vocab,start = 2)}
itosrc = {i:ch for i,ch in enumerate(src_vocab,start = 2)}

targtoi = {ch:(i) for i,ch in enumerate(trg_vocab,start = 4)}
srctoi["<PAD>"] = 0
targtoi["<PAD>"] = 0
targtoi["<SOS>"] = 1
targtoi["<EOS>"] = 2
srctoi["<UNK>"] = 1
targtoi["<UNK>"] = 3

encode_src = lambda s: [srctoi[i] for i in s.split()]
encode_tar = lambda s: [targtoi[i] for i in s.split()]
decode_tar = lambda d: " ".join([itotarg[i] for i in d])
itotarg = dict(map(reversed,targtoi.items()))


In [7]:
input_data = []
target = []
for src, targ in sentences:
    input_data.append(torch.tensor(encode_src(src),dtype = torch.long))
    target.append(torch.tensor(encode_tar(targ),dtype = torch.long))


In [9]:
class SelfAttention(nn.Module):
    def __init__(self,embed_dim,heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.heads = heads
        self.head_dim = embed_dim // heads

        self.W_q = nn.Linear(embed_dim,embed_dim)
        self.W_k = nn.Linear(embed_dim,embed_dim)
        self.W_v = nn.Linear(embed_dim,embed_dim)
        self.fc_out = nn.Linear(embed_dim,embed_dim)

    def forward(self,query,key,value,mask):

        N = query.shape[0]
        query_len, value_len , key_len = query.shape[1],value.shape[1], key.shape[1]

        query = self.W_q(query)
        key = self.W_k(key)
        value = self.W_v(value)

        # shape -> (Batch, Sequence_leng, heads, head_dim) 
        query = query.reshape(N,query_len,self.heads,self.head_dim)
        key = key.reshape(N,key_len,self.heads,self.head_dim)
        value = value.reshape(N,value_len,self.heads,self.head_dim)

        # shape -> (Batch, heads,Sequence_leng, head_dim)  (0,1,2,3)        
        query = query.transpose(1,2)
        key = key.transpose(1,2)
        value = value.transpose(1,2)

        # shape -> (Batch, heads, (T)query_len, (T)key_len)
        energy = torch.matmul(query, key.transpose(-2,-1))

        if mask is not None:
            energy = energy.masked_fill(mask == 0,-1e20)

        # shape -> (Batch, heads, (T)query_len, (T)key_len) 
        # shape -> (0,1,2,3) now dim = 2 denotes - query row wise and dim = 3 denotes columns wise key.
        weight_probs = torch.softmax(energy/(self.head_dim**0.5),dim = 3)

        # (Batch, heads, Query_len, key_len)
        attention = torch.matmul(weight_probs,value).reshape(N,query_len,self.heads*self.head_dim)
        
        #  (Batch, heads, sequence_len * head_dim)
        out = self.fc_out(attention)
        return out        

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self,embed_dim,heads,dropout,forward_dim):
        super().__init__()
        self.attention = SelfAttention(embed_dim,heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim,forward_dim*embed_dim),
            nn.ReLU(),
            nn.Linear(forward_dim*embed_dim,embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self,query,key,value,mask):
        N = query.shape[0]
        attention = self.attention(query,key,value,mask)
        normalized_out = self.dropout(self.norm1(attention+query))

        ff_out = self.feed_forward(normalized_out)
        out = self.norm2(ff_out + normalized_out)

        return out

In [11]:
class PositionalEncoding(nn.Module):
    def __init__(self,max_length,embed_dim):
        super().__init__()

        pos_vector = torch.zeros((max_length,embed_dim))

        # [0,1,2,3] => [[0],[1],[2],[3]]
        pos = torch.arange(start = 0, end= max_length, step = 1).float().unsqueeze(1)

        index = torch.arange(start = 0,end = embed_dim,step = 2 ).float()

        div_term = 1/torch.tensor(10000.0) ** (index/embed_dim)
        
        pos_vector[:,0::2] = torch.sin(pos * div_term)
        pos_vector[:,1::2] = torch.cos(pos * div_term)

        self.register_buffer("pos_vector",pos_vector)

    def forward(self,word_embeds):

        # shape (B,T,C) so, word_embeds.size(1) =  T(sequence_length)
        return word_embeds + self.pos_vector[:word_embeds.size(1)]     
        

In [12]:
class Encoder(nn.Module):
    def __init__(self,vocab_size, embed_dim, heads, dropout, num_layers, forward_dim, max_length):
        super().__init__()
        self.word_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = PositionalEncoding(max_length,embed_dim)
        
        self.encoder_layers = nn.ModuleList(
            [TransformerBlock(embed_dim,heads,dropout,forward_dim) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self,x,mask):

        embeddings = self.word_embedding(x)
        pos_embeddings = self.dropout(self.pos_encoding(embeddings))

        for layer in self.encoder_layers:
            out = layer(pos_embeddings,pos_embeddings,pos_embeddings,mask)

        return out    

In [13]:
class DecoderBlock(nn.Module):
    def __init__(self,embed_dim,heads,dropout,forward_exp):
        super().__init__()
        self.attention = SelfAttention(embed_dim,heads)
        self.norm = nn.LayerNorm(embed_dim)
        self.transformer_block = TransformerBlock(embed_dim,heads,dropout,forward_exp)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x,enc_out,src_mask,trg_mask):
        
        masked_attention = self.attention(x,x,x,trg_mask)
        norm_out = self.norm(x+masked_attention)
        norm_out = self.dropout(norm_out)

        final_out = self.transformer_block(norm_out,enc_out,enc_out,src_mask)

        return final_out
        

In [14]:
class Decoder(nn.Module):
    def __init__(self,src_vocab_len,num_layers, targ_vocab_len , embed_dim, heads,max_length,dropout,
                 forward_dim):
        super().__init__()
        
        self.embed_dim = embed_dim

        self.embeds = nn.Embedding(targ_vocab_len,embed_dim)
        self.pos_encods = PositionalEncoding(max_length,embed_dim)
        
        self.decoder_layers = nn.ModuleList(
            [DecoderBlock(embed_dim,heads,dropout,forward_dim) for _ in range(num_layers)] 
        )
    
        self.fc_out = nn.Linear(embed_dim,targ_vocab_len)
        
        
    def forward(self,target_inp,enc_out,src_mask,targ_mask):
        word_embeddings = self.embeds(target_inp)

        pos_encodings = self.pos_encods(word_embeddings)

        for layer in self.decoder_layers:
            out = layer(pos_encodings,enc_out,src_mask,targ_mask)

        final_out = self.fc_out(out)
        # probs = torch.softmax(final_out,dim = 2)
        return final_out         
        

In [15]:
class Transformer(nn.Module):
    def __init__(self,src_vocab_size,targ_vocab_size,forward_exp,device,heads=8,embed_dim=512,max_length=100,dropout=0,
                 num_layers=6,src_pad_idx=0,trg_pad_idx=0):
        super().__init__()
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device
        self.encoder = Encoder(src_vocab_size,embed_dim,heads,dropout,num_layers,forward_exp,max_length)
        self.decoder  = Decoder(src_vocab_size,num_layers,targ_vocab_size,embed_dim,heads,max_length,dropout,forward_exp)

    def create_src_mask(self,src):
        # src shape (B,T) => (B,1,1,T)
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask.to(self.device)

    def create_target_mask(self,tar):
        N,seq_len = tar.shape
        trg_mask = torch.tril(torch.ones(seq_len,seq_len)).expand(N,1,seq_len,seq_len)
        return trg_mask.to(self.device)
        
    def forward(self,x,y):
        src_mask = self.create_src_mask(x)
        trg_mask = self.create_target_mask(y)
        encoder_out = self.encoder(x,src_mask)
        decoder_out = self.decoder(y,encoder_out,src_mask,trg_mask)

        return decoder_out

In [16]:
from torch.nn.utils.rnn import pad_sequence

input_data = pad_sequence(input_data,padding_value = 0, batch_first = True)
target = pad_sequence(target,padding_value = 0, batch_first = True)

In [18]:
device =  "cuda"
model = Transformer(len(srctoi),len(targtoi),4,device).to(device)
model

Transformer(
  (encoder): Encoder(
    (word_embedding): Embedding(43, 512)
    (pos_encoding): PositionalEncoding()
    (encoder_layers): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): SelfAttention(
          (W_q): Linear(in_features=512, out_features=512, bias=True)
          (W_k): Linear(in_features=512, out_features=512, bias=True)
          (W_v): Linear(in_features=512, out_features=512, bias=True)
          (fc_out): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (feed_forward): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): ReLU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (dropout): Dropout(p=0, inplace=False)
      )
    )
    (dropout): Dropout(p=0, inplace=False)
  )
  (decoder): Decoder(
    (em

In [23]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(),lr = 0.0001)

In [20]:
dataset = TensorDataset(input_data,target)

In [21]:
train_loader = DataLoader(dataset,batch_size=2,shuffle=True)

In [24]:
epochs = 100
losses = []

model.train()

for epoch in range(epochs):

    total_loss = 0

    for batch, (x, y) in enumerate(train_loader):

        x = x.to(device)
        y = y.to(device)

        # SHIFT TARGETS
        decoder_input = y[:, :-1]
        targets = y[:, 1:]

        # FORWARD PASS
        out = model(x, decoder_input)

        # RESHAPE OUTPUT
        out = out.reshape(-1, out.size(-1))

        # RESHAPE TARGETS
        targets = targets.reshape(-1)

        # LOSS
        loss = criterion(out, targets)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    losses.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch} Loss: {avg_loss:.4f}")

Epoch 0 Loss: 3.5732
Epoch 10 Loss: 0.0431
Epoch 20 Loss: 0.0147
Epoch 30 Loss: 0.0086
Epoch 40 Loss: nan
Epoch 50 Loss: 0.0041
Epoch 60 Loss: nan
Epoch 70 Loss: 0.0025
Epoch 80 Loss: 0.0021
Epoch 90 Loss: 0.0018


In [25]:
def translate(sentence, max_len=10):

    model.eval()

    src = torch.tensor(
        [encode_src(sentence)],
        dtype=torch.long
    ).to(device)

    outputs = [targtoi["<SOS>"]]

    for _ in range(max_len):

        trg = torch.tensor(
            [outputs],
            dtype=torch.long
        ).to(device)

        with torch.no_grad():

            output = model(src, trg)

        next_token = output.argmax(2)[:, -1].item()

        outputs.append(next_token)

        if next_token == targtoi["<EOS>"]:
            break

    translated = [itotarg[idx] for idx in outputs]

    return translated